In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import ADASYN
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import shuffle
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, SimpleRNN, Conv1D, MaxPooling1D, Dense, Dropout, Flatten
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
dataset_a = pd.read_csv('/content/drive/MyDrive/sleep_apnea/new_features/features_of_A.csv')
dataset_b = pd.read_csv('/content/drive/MyDrive/sleep_apnea/new_features/features_of_B.csv')
dataset_c = pd.read_csv('/content/drive/MyDrive/sleep_apnea/new_features/features_of_C.csv')
dataset_x = pd.read_csv('/content/drive/MyDrive/sleep_apnea/new_features/features_of_X.csv')

dataset = pd.concat([dataset_a, dataset_b, dataset_c, dataset_x]).dropna()
X = dataset.drop(columns=["Label", "Record_ID"])
y = dataset["Label"]


# X_train , X_test , y_train , y_test = train_data.drop(columns=["Label", "Record_ID"]), test_data.drop(columns=["Label", "Record_ID"]), train_data["Label"], test_data["Label"]

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X, y, test_size=0.7, random_state=42)


In [ ]:
adasyn = ADASYN(random_state=75)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train_raw, y_train_raw)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)

# X_test_raw, X_val, y_test_raw, y_val = train_test_split(X_test_raw, y_test_raw, test_size=0.5, random_state=42)

X_test_scaled = scaler.transform(X_test_raw)
# X_val = scaler.transform(X_val)

# Encode labels
labelEncoder = LabelEncoder()
y_train_encoded = labelEncoder.fit_transform(y_train_resampled)
y_test_encoded = labelEncoder.transform(y_test_raw)
# y_val = labelEncoder.transform(y_val)


# Reshape for CNN/RNN/LSTM models
X_train_final = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_final = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))
# X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))


In [ ]:
import joblib
from tensorflow.keras.models import load_model

In [ ]:
knn_model = joblib.load('/content/knn_classifier.pkl')
xgb_model = joblib.load('/content/xgb_classifier.pkl')
rf_model = joblib.load('/content/rf_classifier.pkl')
bagging_classifier = joblib.load('/content/bagging_classifier.pkl')
lstm_model = load_model('/content/lstm_model.h5')
rnn_model = load_model('/content/rnn_model.h5')
rnn_cnn_model = load_model('/content/rnn_cnn_model.h5')
rnn_lstm_model = load_model('/content/rnn_lstm_model.h5')

In [ ]:
# Generate predictions for stacking
knn_preds_train = knn_model.predict(X_train_scaled)
xgb_preds_train = xgb_model.predict(X_train_scaled)
rf_preds_train = rf_model.predict(X_train_scaled)
bagging_classifier_preds_train = bagging_classifier.predict(X_train_scaled)

svm_preds_test = knn_model.predict(X_test_scaled)
xgb_preds_test = xgb_model.predict(X_test_scaled)
rf_preds_test = rf_model.predict(X_test_scaled)
bagging_classifier_preds_test = bagging_classifier.predict(X_test_scaled)

lstm_preds_train = (lstm_model.predict(X_train_final) > 0.5).astype(int).flatten()
rnn_preds_train = (rnn_model.predict(X_train_final) > 0.5).astype(int).flatten()
rnn_cnn_preds_train = (rnn_cnn_model.predict(X_train_final) > 0.5).astype(int).flatten()
rnn_lstm_preds_train = (rnn_lstm_model.predict(X_train_final) > 0.5).astype(int).flatten()

lstm_preds_test = (lstm_model.predict(X_test_final) > 0.5).astype(int).flatten()
rnn_preds_test = (rnn_model.predict(X_test_final) > 0.5).astype(int).flatten()
rnn_cnn_preds_test = (rnn_cnn_model.predict(X_test_final) > 0.5).astype(int).flatten()
rnn_lstm_preds_test = (rnn_lstm_model.predict(X_test_final) > 0.5).astype(int).flatten()


407/407 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step
407/407 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
407/407 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
407/407 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


In [ ]:
# Convert X_test_final (3D) to 2D for ML models
X_test_final_ml = X_test_final.reshape((X_test_final.shape[0], X_test_final.shape[1]))

# Generate ML model predictions on the same final test set
knn_preds_test = knn_model.predict(X_test_final_ml)
xgb_preds_test = xgb_model.predict(X_test_final_ml)
rf_preds_test = rf_model.predict(X_test_final_ml)
bagging_classifier_preds_test = bagging_classifier.predict(X_test_final_ml)

# DL model predictions remain the same (already on X_test_final)
lstm_preds_test = (lstm_model.predict(X_test_final) > 0.5).astype(int).flatten()
rnn_preds_test = (rnn_model.predict(X_test_final) > 0.5).astype(int).flatten()
rnn_cnn_preds_test = (rnn_cnn_model.predict(X_test_final) > 0.5).astype(int).flatten()
rnn_lstm_preds_test = (rnn_lstm_model.predict(X_test_final) > 0.5).astype(int).flatten()


# Stacking: Combine predictions to form meta-features
X_meta_test = np.column_stack([knn_preds_test, xgb_preds_test, rf_preds_test,bagging_classifier_preds_test,
                                lstm_preds_test, rnn_preds_test, rnn_cnn_preds_test,rnn_lstm_preds_test])

# Similarly, ensure that ML predictions for training are computed on the same data used for training DL models.
# Typically, you would compute ML predictions on the training data (X_train_scaled) which were used to train ML models.
# If you want to combine on a common set, ensure they are computed on the same data.
# For this snippet, we assume ML predictions for training are already computed consistently.

X_meta_train = np.column_stack([knn_preds_train, xgb_preds_train, rf_preds_train,bagging_classifier_preds_train,
                                lstm_preds_train, rnn_preds_train, rnn_cnn_preds_train,rnn_lstm_preds_train])

# Train meta-model
meta_model = LogisticRegression()
meta_model.fit(X_meta_train, y_train_encoded)

# Get final predictions and evaluate
final_preds = meta_model.predict(X_meta_test)

print("Stacking Model Accuracy:(Logistic Regression)", accuracy_score(y_test_encoded, final_preds))
print("Stacking Model Report:\n", classification_report(y_test_encoded, final_preds))


749/749 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
Stacking Model Accuracy:(Logistic Regression) 0.900747733823468
Stacking Model Report:
               precision    recall  f1-score   support

           0       0.84      0.91      0.88      9091
           1       0.94      0.89      0.92     14848

    accuracy                           0.90     23939
   macro avg       0.89      0.90      0.90     23939
weighted avg       0.90      0.90      0.90     23939



In [ ]:
joblib.dump(meta_model, 'meta_model.pkl')

['meta_model.pkl']

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train Random Forest as the meta-model
meta_model_rf = RandomForestClassifier(n_estimators=200, random_state=42)
meta_model_rf.fit(X_meta_train, y_train_encoded)

final_preds_rf = meta_model_rf.predict(X_meta_test)

print("Stacking Model (Random Forest Meta-Model) Accuracy:", accuracy_score(y_test_encoded, final_preds_rf))
print("Stacking Model Report:\n", classification_report(y_test_encoded, final_preds_rf))

Stacking Model (Random Forest Meta-Model) Accuracy: 0.897781862233176
Stacking Model Report:
               precision    recall  f1-score   support

           0       0.83      0.92      0.87      9091
           1       0.95      0.89      0.91     14848

    accuracy                           0.90     23939
   macro avg       0.89      0.90      0.89     23939
weighted avg       0.90      0.90      0.90     23939



In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Train XGBoost as the meta-model
meta_model_xgb = XGBClassifier(n_estimators=200, learning_rate=0.01, random_state=60)
meta_model_xgb.fit(X_meta_train, y_train_encoded)

# Get final predictions from the meta-model
final_preds_xgb = meta_model_xgb.predict(X_meta_test)

# Evaluate XGBoost Stacking Model
print("Stacking Model (XGBoost Meta-Model) Accuracy:", accuracy_score(y_test_encoded, final_preds_xgb))
print("Stacking Model Report:\n", classification_report(y_test_encoded, final_preds_xgb))


Stacking Model (XGBoost Meta-Model) Accuracy: 0.8993692301265717
Stacking Model Report:
               precision    recall  f1-score   support

           0       0.83      0.92      0.87      9091
           1       0.95      0.89      0.92     14848

    accuracy                           0.90     23939
   macro avg       0.89      0.90      0.90     23939
weighted avg       0.90      0.90      0.90     23939



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

meta_model_gbs = GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, random_state=60)
meta_model_gbs.fit(X_meta_train, y_train_encoded)

final_preds_gbs = meta_model_gbs.predict(X_meta_test)

print("Stacking Model (Gradient Boosting Meta-Model) Accuracy:", accuracy_score(y_test_encoded, final_preds_gbs))
print("Stacking Model Report:\n", classification_report(y_test_encoded, final_preds_gbs))

Stacking Model (Gradient Boosting Meta-Model) Accuracy: 0.8996616400016709
Stacking Model Report:
               precision    recall  f1-score   support

           0       0.83      0.92      0.87      9091
           1       0.95      0.89      0.92     14848

    accuracy                           0.90     23939
   macro avg       0.89      0.90      0.90     23939
weighted avg       0.90      0.90      0.90     23939



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

meta_model_dt = DecisionTreeClassifier(max_depth = 3,criterion='entropy',random_state=42)
meta_model_dt.fit(X_meta_train, y_train_encoded)

final_preds_dt = meta_model_dt.predict(X_meta_test)

print("Stacking Model (Decision Tree Meta-Model) Accuracy:", accuracy_score(y_test_encoded, final_preds_dt))
print("Stacking Model Report:\n", classification_report(y_test_encoded, final_preds_dt))


Stacking Model (Decision Tree Meta-Model) Accuracy: 0.900371778269769
Stacking Model Report:
               precision    recall  f1-score   support

           0       0.84      0.91      0.87      9091
           1       0.94      0.89      0.92     14848

    accuracy                           0.90     23939
   macro avg       0.89      0.90      0.90     23939
weighted avg       0.90      0.90      0.90     23939



In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

def weighted_voting(predictions_proba_list, weights):
    """Weighted soft voting for ensemble"""
    predictions_proba_list = np.array(predictions_proba_list)  # Ensure array format
    weighted_probs = np.average(predictions_proba_list, axis=0, weights=weights)  # Average along model axis
    return np.argmax(weighted_probs, axis=1)  # Get class with highest probability

# Get probabilities from ML models

xgb_probs = xgb_model.predict_proba(X_test_scaled)  # (n_samples, 2)
rf_probs = rf_model.predict_proba(X_test_scaled)  #  (n_samples, 2)
knn_probs = knn_model.predict_proba(X_test_scaled)
bagging_classifier_probs = bagging_classifier.predict_proba(X_test_scaled)  # (n_samples, 2)

# Get probabilities from DL models (originally (n_samples, 1))
lstm_probs = lstm_model.predict(X_test_final).flatten()
rnn_probs = rnn_model.predict(X_test_final).flatten()
rnn_cnn_probs = rnn_cnn_model.predict(X_test_final).flatten()
rnn_lstm_probs = rnn_lstm_model.predict(X_test_final).flatten()

# Convert DL probabilities to (n_samples, 2) to match ML model outputs
lstm_probs = np.column_stack([1 - lstm_probs, lstm_probs])
rnn_probs = np.column_stack([1 - rnn_probs, rnn_probs])
rnn_cnn_probs = np.column_stack([1 - rnn_cnn_probs, rnn_cnn_probs])
rnn_lstm_probs = np.column_stack([1 - rnn_lstm_probs, rnn_lstm_probs])

# **Ensure all arrays have the same number of samples before stacking**
n_samples = min(len(xgb_probs), len(lstm_probs), len(rnn_probs), len(rnn_cnn_probs),len(rnn_lstm_probs),)  # Find min sample count

# Trim all arrays to the same size (handle small mismatches)
knn_probs = knn_probs[:n_samples]
xgb_probs = xgb_probs[:n_samples]
rf_probs = rf_probs[:n_samples]
bagging_classifier_probs = bagging_classifier_probs[:n_samples]
lstm_probs = lstm_probs[:n_samples]
rnn_probs = rnn_probs[:n_samples]
rnn_cnn_probs = rnn_cnn_probs[:n_samples]
rnn_lstm_probs = rnn_lstm_probs[:n_samples]
y_test_final_trimmed = y_test_encoded[:n_samples]  # Ensure labels also match

# Stack probabilities correctly (shape: (n_models, n_samples, 2))
all_probs = np.array([knn_probs, xgb_probs, rf_probs,bagging_classifier_probs, lstm_probs, rnn_probs, rnn_cnn_probs,rnn_lstm_probs])

weights = np.array([0.09, 0.12, 0.15, 0.10, 0.11, 0.09, 0.13, 0.1])  # Ensure weights sum to 1

# Apply Weighted Voting
final_preds_weighted = weighted_voting(all_probs, weights)

# Evaluate Weighted Voting Model
print("Weighted Voting Accuracy:", accuracy_score(y_test_final_trimmed, final_preds_weighted))
print("Weighted Voting Report:\n", classification_report(y_test_final_trimmed, final_preds_weighted))


749/749 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step
749/749 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
Weighted Voting Accuracy: 0.8771043067797318
Weighted Voting Report:
               precision    recall  f1-score   support

           0       0.79      0.91      0.85      9091
           1       0.94      0.85      0.90     14848

    accuracy                           0.88     23939
   macro avg       0.87      0.88      0.87     23939
weighted avg       0.89      0.88      0.88     23939



In [ ]:
knn_model = joblib.load("knn_classifier.pkl")
xgb_model = joblib.load("xgb_classifier.pkl")
rf_model = joblib.load("rf_classifier.pkl")
bagging_classifier = joblib.load("bagging_classifier.pkl")

lstm_model = load_model("lstm_model.h5")
rnn_model = load_model("rnn_model.h5")
rnn_cnn_model = load_model("rnn_cnn_model.h5")
rnn_lstm_model = load_model("rnn_lstm_model.h5")

meta_model = joblib.load("meta_model.pkl")

In [ ]:
def stacked_model_prediction(feature_df):
    """Runs predictions using ML/DL models and meta-classifier."""
    sc = StandardScaler()
    feature_array = sc.fit_transform(feature_df)

    feature_array_reshaped = feature_array.reshape((feature_array.shape[0], feature_array.shape[1], 1))

    # ML Model Predictions
    knn_preds = knn_model.predict(feature_array)
    print("KNN")
    xgb_preds = xgb_model.predict(feature_array)
    print("XGB")
    rf_preds = rf_model.predict(feature_array)
    print("rf")
    bagging_preds = bagging_classifier.predict(feature_array)
    print("ML Done")
    # DL Model Predictions
    lstm_preds = (lstm_model.predict(feature_array_reshaped) > 0.5).astype(int).flatten()
    rnn_preds = (rnn_model.predict(feature_array_reshaped) > 0.5).astype(int).flatten()
    rnn_cnn_preds = (rnn_cnn_model.predict(feature_array_reshaped) > 0.5).astype(int).flatten()
    rnn_lstm_preds = (rnn_lstm_model.predict(feature_array_reshaped) > 0.5).astype(int).flatten()

    # Stacking: Combine predictions
    X_meta = np.column_stack([knn_preds, xgb_preds, rf_preds, bagging_preds,
                               lstm_preds, rnn_preds, rnn_cnn_preds, rnn_lstm_preds])

    final_pred = meta_model.predict(X_meta)
    return final_pred

In [ ]:
x = dataset[dataset['Record_ID']=='a01'].drop(columns=['Label','Record_ID'])

In [ ]:
sum(stacked_model_prediction(x))

KNN
XGB
rf
ML Done
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


306